# BizInsight Agent: Multi-Agent Business Intelligence System

## Kaggle Capstone Project — Agents for Business Track

### Course Concepts Demonstrated

| Day | Concept | Implementation |
|-----|---------|----------------|
| **Day 1** | Multi-Agent Systems | 4 CrewAI agents (Orchestrator, Data Analyst, Insight Generator, Report Writer) |
| **Day 1** | Context Engineering | AGENTS.md provides project context; Skills loaded on-demand |
| **Day 2** | MCP Servers | 2 custom MCP servers (data access + chart generation) with 6 tools |
| **Day 3** | Agent Skills | 3 SKILL.md files with Progressive Disclosure pattern |
| **Day 4** | Security | Sandboxed file I/O, input validation, egress governance || **Day 5** | Spec-Driven Dev | BDD specs, Policy Server (Zero-Trust), Context Hygiene |

## 1. Project Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools mcp pandas numpy matplotlib seaborn pyyaml python-dotenv

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('Environment ready!')

## 2. Load Sample Business Data

In [ ]:
# Load the sample dataset
df = pd.read_csv('/kaggle/working/bizinsight-agent/data/sample_business_data.csv')
print(f'Dataset: {len(df)} rows x {len(df.columns)} columns')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

## 3. Day 2 Demo: MCP Data Server Tools

The MCP (Model Context Protocol) servers provide standardized tool access.
In this notebook, we demonstrate the same tools directly.

In [ ]:
# MCP Tool: get_columns — Column profiling
def get_columns(df):
    info = []
    for col in df.columns:
        info.append({
            'name': col, 'dtype': str(df[col].dtype),
            'null_count': int(df[col].isnull().sum()),
            'unique_count': int(df[col].nunique()),
            'sample_values': df[col].dropna().head(3).tolist()
        })
    return json.dumps({'total_rows': len(df), 'columns': info}, indent=2)

print(get_columns(df))

In [ ]:
# MCP Tool: compute_aggregate — Sales by Category
def compute_aggregate(df, group_by, metric, agg='sum'):
    result = df.groupby(group_by)[metric].agg(agg).reset_index()
    result.columns = [group_by, f'{agg}_of_{metric}']
    return result.sort_values(f'{agg}_of_{metric}', ascending=False)

sales_by_cat = compute_aggregate(df, 'category', 'sales', 'sum')
print('Sales by Category:')
print(sales_by_cat.to_string(index=False))

In [ ]:
# MCP Tool: detect_anomalies — IQR method
def detect_anomalies_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return {
        'method': 'IQR', 'q1': float(q1), 'q3': float(q3), 'iqr': float(iqr),
        'lower_bound': float(lower), 'upper_bound': float(upper),
        'anomaly_count': int(mask.sum()),
        'anomaly_values': series[mask].tolist()
    }

for col in ['sales', 'profit', 'customer_rating']:
    result = detect_anomalies_iqr(pd.to_numeric(df[col], errors='coerce'))
    print(f'\nAnomalies in {col}:')
    print(f'  Method: {result["method"]}')
    print(f'  IQR: {result["iqr"]:.2f}')
    print(f'  Bounds: [{result["lower_bound"]:.2f}, {result["upper_bound"]:.2f}]')
    print(f'  Anomalies found: {result["anomaly_count"]}')

## 4. Day 2 Demo: MCP Charts Server Tools

In [ ]:
# MCP Tool: create_bar_chart — Sales by Category
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
categories = sales_by_cat['category'].tolist()
sales = sales_by_cat['sum_of_sales'].tolist()
bars = ax.bar(categories, sales, color=['#4A90D9', '#2ECC71', '#E74C3C', '#F39C12'], edgecolor='white')
ax.set_title('Total Sales by Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Sales ($)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.01,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=10)
plt.savefig('/kaggle/working/sales_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MCP Tool: create_line_chart — Sales trend over time
df['date'] = pd.to_datetime(df['date'])
monthly = df.groupby(df['date'].dt.to_period('M'))['sales'].sum().reset_index()
monthly['date'] = monthly['date'].astype(str)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(monthly['date'], monthly['sales'], color='#2ECC71', linewidth=2, marker='o', markersize=6)
ax.fill_between(range(len(monthly)), monthly['sales'], alpha=0.1, color='#2ECC71')
ax.set_title('Monthly Sales Trend', fontsize=14, fontweight='bold')
ax.set_ylabel('Sales ($)')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.savefig('/kaggle/working/sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Day 3 Demo: Agent Skills — Progressive Disclosure

Skills load in 3 levels:
1. **Metadata** (name + description) — always in context (~200 tokens)
2. **SKILL.md body** — loaded when skill triggers (~800 tokens)
3. **Scripts/References** — loaded on demand (0 tokens in context)

In [ ]:
# Level 1: Show skill metadata (what the agent sees on every turn)
import yaml

skills_dir = '/kaggle/working/bizinsight-agent/skills'
for skill_name in ['data-analysis', 'insight-generation', 'report-writing']:
    skill_md = os.path.join(skills_dir, skill_name, 'SKILL.md')
    with open(skill_md) as f:
        content = f.read()
    # Extract YAML frontmatter
    end = content.index('---', 3)
    frontmatter = content[3:end].strip()
    print(f'=== {skill_name} ===')
    print(frontmatter)
    print(f'Token estimate (metadata): ~{len(frontmatter.split()) * 1.3:.0f} tokens')
    print()

In [ ]:
# Level 3: Scripts execute without polluting the token window
import sys
sys.path.insert(0, '/kaggle/working/bizinsight-agent')
from skills.data_analysis.scripts.analyze import (
    calculate_profit_margin, calculate_growth_rate,
    compute_concentration_ratio, summarize_numeric_series
)

# Example: Profit margin
print(f'Profit margin (12,500 sales, 5,000 profit): {calculate_profit_margin(12500, 5000)}%')

# Example: Growth rate
print(f'Growth rate (15,000 -> 18,500): {calculate_growth_rate(18500, 15000)}%')

# Example: Concentration ratio
items = [{'cat': 'Electronics', 'sales': 300000}, {'cat': 'Clothing', 'sales': 80000},
         {'cat': 'Food', 'sales': 30000}, {'cat': 'Home', 'sales': 20000}]
cr = compute_concentration_ratio(items, 'sales', top_n=2)
print(f'Top-2 concentration ratio: {cr}% (top 2 categories drive {cr}% of sales)')

## 6. Day 1 Demo: Multi-Agent Pipeline Architecture

The system uses CrewAI to orchestrate 3 agents in a DAG:
```
Data Analyst → Insight Generator → Report Writer
     ↓                ↓                    ↓
  (MCP Data       (Business           (MCP Charts
   Server)         Frameworks)          Server)
```

In [ ]:
# Show agent configurations
import yaml
with open('/kaggle/working/bizinsight-agent/crew/config/agents.yaml') as f:
    agents = yaml.safe_load(f)

for name, config in agents.items():
    print(f'**{config["role"]}**')
    print(f'  Goal: {config["goal"][:80]}...')
    print(f'  Backstory length: {len(config["backstory"])} chars')
    print()

## 7. Day 4 Demo: Security Features

| Security Pillar | Implementation |
|-----------------|----------------|
| **Sandboxing** | File I/O restricted to `data/` and `output/` directories |
| **Egress Governance** | MCP servers have zero outbound network access |
| **Input Validation** | All tool parameters validated before execution |
| **Zero Ambient Authority** | Each tool has minimal required permissions |
| **Path Traversal Prevention** | `_safe_path()` blocks `../` access |

In [ ]:
# Demonstrate security: path traversal prevention
import os

DATA_DIR = '/kaggle/working/bizinsight-agent/data'

def safe_path(filename, base_dir):
    """Day 4: Prevent directory traversal attacks."""
    full = os.path.abspath(os.path.join(base_dir, filename))
    if not full.startswith(os.path.abspath(base_dir)):
        raise ValueError(f'ACCESS DENIED: {filename} escapes allowed directory')
    return full

# Test allowed path
print('Allowed:', safe_path('sample_business_data.csv', DATA_DIR))

# Test blocked path traversal
try:
    safe_path('../../../etc/passwd', DATA_DIR)
except ValueError as e:
    print(f'Blocked: {e}')

## 8. Run Tests (Eval Coverage — Day 3)

36 tests covering MCP servers, Agent Skills, and CrewAI pipeline.

In [ ]:
!cd /kaggle/working/bizinsight-agent && python -m pytest tests/ -v --tb=short 2>&1 | tail -50

## 9. Concept Mapping Summary

This project demonstrates **5 key concepts** from the 5-day course:

### Day 1: The New SDLC with Vibe Coding
- **Multi-Agent System**: 4 specialized CrewAI agents with role-based delegation
- **Orchestrator Pattern**: Orchestrator agent breaks user queries into sub-tasks
- **Context Engineering**: AGENTS.md provides project-level context to all agents
- **Harness Engineering**: MCP servers + Skills form the agent's harness

### Day 2: Agent Tools & Interoperability
- **MCP Servers**: 2 custom servers with stdio transport (Discovery → Configuration → Connection)
- **NxM Problem Solved**: Multiple agents share the same MCP tools — O(N+M) integration
- **MCP Best Practices**: No hardcoded credentials, scoped file access, read-only data

### Day 3: Agent Skills
- **SKILL.md Format**: YAML frontmatter + markdown instructions
- **Progressive Disclosure**: 3-level loading (metadata → body → resources)
- **DAG Orchestration**: Tasks flow as Analyze → Insights → Report
- **Eval Coverage**: 36 tests verifying skill triggering, output quality, structure

### Day 4: Security & Evaluation
- **Sandboxing**: File I/O restricted to designated directories
- **Egress Governance**: No outbound network access from MCP servers
- **Input Validation**: All tool parameters validated
- **Path Traversal Prevention**: `_safe_path()` blocks directory escapes

### Day 5: Spec-Driven Production Grade Development
- **Spec-Driven Development (SDD)**: BDD specs in `specs/` using Gherkin (Given/When/Then)
- **Policy Server**: Zero-Trust guardrail with YAML-based structural gating (role/environment)
- **Context Hygiene**: PII masking, `[[VARIABLE]]` placeholder resolution, prompt sanitization
- **AI-Generated Test Coverage**: 67 tests covering all 5 days' concepts